# 실제 유저 추천 데모

유저 ID와 모델을 넣으면 그 유저에게 추천할 게임 Top-N을 보여준다.
평가(NDCG)가 아니라 **실제 추천 결과 확인용** — 전체 데이터로 돌리고(hold-out 없음),
이미 보유한 게임은 제외한다.

모델은 자유롭게 갈아끼울 수 있다: `popularity / content / user_cf / item_cf / svd / mab / hybrid`

## 1. 준비 (한 번만)

In [ ]:
import sys; sys.path.insert(0, "../src")
import pandas as pd
import data
from models import popularity, content, user_cf, item_cf, svd, mab

ctx = data.load_context("../data/processed")   # 전체 데이터
print("유저", len(ctx["taste"]), "명 | 게임", len(ctx["all_games"]), "개")

## 2. 추천 함수 (모델 갈아끼우기 가능)

In [ ]:
SINGLE = {"popularity": popularity, "content": content, "user_cf": user_cf,
          "item_cf": item_cf, "svd": svd, "mab": mab}

def _mm(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

def get_scores(model_name, user, alpha=0.85):
    """model_name: 위 6개 + 'hybrid'(item_cf*alpha + content*(1-alpha))"""
    if model_name == "hybrid":
        ic = _mm(item_cf.recommend(user, ctx))
        ct = _mm(content.recommend(user, ctx))
        return alpha * ic + (1 - alpha) * ct
    return SINGLE[model_name].recommend(user, ctx)

def recommend(user, model_name="item_cf", n=10, alpha=0.85):
    scores = get_scores(model_name, user, alpha)
    owned = ctx["owned"].get(user, set())                       # 보유 게임 제외
    scores = scores.drop(index=[a for a in owned if a in scores.index], errors="ignore")
    top = scores.nlargest(n)
    return pd.DataFrame({"game": [ctx["names"].get(a, a) for a in top.index],
                         "score": top.values.round(3)})

## 3. 사용 — 유저·모델만 바꾸면 됨

In [ ]:
USER = list(ctx["taste"].index)[0]     # 원하는 steamid로 바꾸기

# 이 유저가 실제로 많이 한 게임 (참고)
played = ctx["interactions"]
top_played = played[played["steamid"] == USER].nlargest(5, "w")["game_name"].tolist()
print("이 유저가 많이 한 게임:", top_played)

# 모델 바꿔가며 추천 보기  ← "item_cf"를 hybrid/content/svd 등으로 교체
recommend(USER, "item_cf", n=10)

## 4. 여러 모델 한눈에 비교 (같은 유저)

In [ ]:
for m in ["item_cf", "hybrid", "content", "user_cf", "svd", "popularity"]:
    top5 = recommend(USER, m, n=5)["game"].tolist()
    print(f"[{m:>10}]", top5)

## 참고
- **추천 = 전체 데이터 기반** (평가와 달리 hold-out 안 함). 유저가 안 해본 게임 중 Top-N.
- **모델 교체**: `recommend(USER, "hybrid")` 처럼 이름만 바꾸면 됨.
- **하이브리드 비중**: `recommend(USER, "hybrid", alpha=0.7)` — alpha↑ = item_cf 비중↑.
- `svd`는 첫 호출 때 학습(캐싱), `mab`는 유저마다 계산.